# 03 - Generation and Safety (مطابق لـ `generation_safety.py`)

بنذاكر هنا الـ safety gate وتجهيز generation messages قبل إرسال أي حاجة للـ Groq.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'rag' else Path.cwd()))
import asyncio
import inspect
from rag import generation_safety
from rag.retrieval import retrieve
print('Production module:', generation_safety.__file__)
print('Blocked phrases:', generation_safety.UNSAFE)

## 1. Safety check
السؤال الخطير يتوقف قبل ما يوصل للـ external AI provider.

In [ ]:
print(inspect.getsource(generation_safety.safety_check))
for question in ['What is S34?', 'How to forge an ancient artifact?']:
    print(question, '=>', generation_safety.safety_check(question))

## 2. Generate بدون provider
لو `chat_call=None`، الدالة ترجع messages فقط علشان نراجع الأدلة قبل الاتصال بـ Groq.

In [ ]:
print(inspect.getsource(generation_safety.generate))
question = 'What is S34?'
records = retrieve(question, codes=['S34'])
packet = asyncio.run(generation_safety.generate(question, records))
print('Blocked:', packet['blocked'])
print('Message roles:', [m['role'] for m in packet['messages']])
print('System instruction preview:', packet['messages'][0]['content'][:220])

## الخلاصة
`generation_safety.py` يحرس السؤال ويجهز prompt؛ أما الاتصال الفعلي واختيار الموديل ففي `chatbot.py`.